In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sirs_gillespie(
    beta0,   # mean transmission rate
    gamma,   # recovery rate
    mu,      # birth/death rate
    delta,   # waning immunity rate
    dbeta,   # seasonal amplitude of beta(t)
    S0, I0, R0,  # initial S,I,R
    T_max=520    # max time (weeks)
):
    
    t = 0.0
    S, I, R = S0, I0, R0
    
    t_values = [t]
    S_values = [S]
    I_values = [I]
    R_values = [R]
    
    # State-updates for each possible event:
    # (dS, dI, dR) in the same order as the rates
    updates = [
        ( +1,  0,  0),  # Birth
        ( -1,  0,  0),  # Death of S
        (  0, -1,  0),  # Death of I
        (  0,  0, -1),  # Death of R
        ( -1, +1,  0),  # Infection S->I
        (  0, -1, +1),  # Recovery I->R
        ( +1,  0, -1)   # Waning R->S
    ]
    
    while t < T_max:
        N = S + I + R
        if N <= 0:  # infection died out (which we currently rule out -- see below)
            break
        
        # Seasonal transmission
        beta_t = beta0 * (1.0 + dbeta * np.sin(2.0*np.pi*t/52.0))
        
        # Reaction rates
        birth_rate     = mu * N
        s_death_rate   = mu * S
        i_death_rate   = mu * I
        r_death_rate   = mu * R
        infection_rate = beta_t * S * I / N if N > 0 else 0.0
        recovery_rate  = gamma * I
        waning_rate    = delta * R
        
        rates = [
            birth_rate,
            s_death_rate,
            i_death_rate,
            r_death_rate,
            infection_rate,
            recovery_rate,
            waning_rate
        ]
        
        total_rate = sum(rates)
        if total_rate <= 0:
            break
        
        # Time to next event
        r1 = np.random.random()
        dt = -np.log(r1) / total_rate
        t_new = t + dt
        if t_new > T_max:
            # If next event is beyond T_max, stop
            break
        
        t = t_new
        
        # Determine which event occurs
        r2 = np.random.random() * total_rate
        cumulative = 0.0
        event_idx = None
        for i, rate in enumerate(rates):
            cumulative += rate
            if r2 < cumulative:
                event_idx = i
                break
        
        # Apply that event
        dS, dI, dR = updates[event_idx]
        S += dS
        I += dI
        R += dR

        # Avoid complete dying-out:
        I = max(1,I)
        
        # Record state
        if t-t_values[-1] > 1/7: # Avoid excessive point density
            t_values.append(t)
            S_values.append(S)
            I_values.append(I)
            R_values.append(R)
            #if int(t_values[-1]/52) > int(t_values[-2]/52):
            #    print("t/52:", t/52)
            
    
    return (
        np.array(t_values),
        np.array(S_values),
        np.array(I_values),
        np.array(R_values)
    )



In [ ]:
# Read in parameter values from fit:

In [ ]:
import glob

csv_files = glob.glob('stan_output/sinusoid_2025/*20250801092152*.csv')  # dt=1/8

from cmdstanpy import CmdStanModel
from cmdstanpy import from_csv


fit = from_csv(csv_files)

def DiscTimeRate(r, dt):
    return (1 - np.exp(-r*dt))/dt

import re

def best_draw_to_dict(row):
    """
    Convert a series of Stan draws into a structured dict,
    grouping vector parameters like param[1], param[2], ... into arrays.
    """
    result = {}
    for col in row.index:
        # Attempt to match something like "paramName[123]"
        m = re.match(r"(.*)\[(\d+)\]$", col)
        if m:
            base_name = m.group(1)
            idx = int(m.group(2))  # 1-based index from Stan
            val = row[col]

            if base_name not in result:
                # store them in a dict first, convert to array after.
                result[base_name] = {}
            result[base_name][idx] = val
        else:
            # it is a scalar param or a special column like chain__, iter__, ...
            result[col] = row[col]

    # Convert any dict-of-indices to a list or array
    # Stan uses 1-indexing
    for k, v in list(result.items()):
        if isinstance(v, dict):
            # v is a dictionary of indices -> values
            max_idx = max(v.keys())
            # create an array of length = max index
            arr = np.empty(max_idx, dtype=float)
            for i in range(1, max_idx + 1):
                arr[i - 1] = v[i]  # shift to zero-indexing
            result[k] = arr
    return result

df_draws = fit.draws_pd()
lp = df_draws["lp__"]  # log posterior column
imax = np.argmax(lp)  # index of sample with highest lp__
best_draw = df_draws.iloc[imax]

best_draw_dict = best_draw_to_dict(best_draw)

dt_discrete = 1.0/8

mu = DiscTimeRate(1/(52 * 80.0), dt_discrete)


beta0 = DiscTimeRate(best_draw_dict['beta0'], dt_discrete)
betaphase = best_draw_dict['betaphase']
gamma = DiscTimeRate(best_draw_dict['gamma'], dt_discrete)
delta = DiscTimeRate(best_draw_dict['delta'], dt_discrete)
dbeta = DiscTimeRate(best_draw_dict['dbeta'], dt_discrete)
s0 = best_draw_dict['S0']
i0 = best_draw_dict['I0']

In [ ]:


#beta0  = 0.713929532
#gamma  = 1/2.5
#mu     = 1/(80*52)
#delta  = 0.0017271
#dbeta  = 0.18569449

# Initial population

#np.random.seed(2)
#N0 = int(2e4)

np.random.seed(4)
N0 = int(2.5e5)

#np.random.seed(2)
#N0 = int(2e6)

#np.random.seed(2)
#N0 = int(2e7)

I0 = int(i0*N0)
R0 = int(s0*N0)
S0 = N0 - I0 - R0

T_max = 150*52  # weeks

# Run simulation
t_vals, S_vals, I_vals, R_vals = sirs_gillespie(
    beta0, gamma, mu, delta, dbeta,
    S0, I0, R0,
    T_max
)

In [ ]:
# Plot
plt.figure(figsize=(3.5,2), dpi=200)

t_plot = np.array(t_vals)/52
I_plot = np.array(I_vals)/N0

plot_mask = t_plot >= 100
t_plot = t_plot[plot_mask]
I_plot = I_plot[plot_mask]

#plt.plot(t_vals, S_vals/N0, label='S', color='blue')
#plt.plot(np.array(t_vals)[-10000000:]/52, np.array(I_vals)[-10000000:]/N0, label='', color='red')
plt.plot(t_plot, I_plot, label=r'$I(t)$, ' + r"$N_0=$" + f'{N0:,}', color=plt.cm.inferno(0.4))
#plt.plot(t_vals, R_vals/N0, label='R', color='green')
plt.xlabel('Time (years)')
plt.ylabel('Prevalence')
#plt.title(f'Seasonal SIRS (Gillespie, ' + r"$N_0=$" + f'{N0:,})')
plt.legend(loc='upper left')
plt.grid(False)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.ylim([-0.001, 0.03])
plt.show()


In [ ]:
from scipy import signal
from joblib import Parallel, delayed
from multiprocessing import Pool

import os
import time

N0 = int(5.0e5)

I0 = int(0.005*N0)
R0 = int(0.35*N0)
S0 = N0 - I0 - R0

T_max = 150*52  # weeks

def compute_psd_from_sim(beta0, gamma, mu, delta, dbeta,
                         S0, I0, R0, T_max, N0,
                         dt=1/52, transient=50):
    
    np.random.seed(int(time.time() * 1e6) % (2**32 - 1) + os.getpid())
    
    # run Gillespie simulation
    t_vals, S_vals, I_vals, R_vals = sirs_gillespie(
        beta0, gamma, mu, delta, dbeta,
        S0, I0, R0, T_max
    )

    # convert to years and normalize by N0
    t_plot = np.array(t_vals)/52
    I_plot = np.array(I_vals)/N0

    # discard transient
    mask = t_plot >= transient
    t_plot = t_plot[mask]
    I_plot = I_plot[mask]

    # regular grid
    t_reg = np.arange(t_plot[0], t_plot[-1], dt)
    I_reg = np.interp(t_reg, t_plot, I_plot)

    print("Final t, I:", t_vals[-1], I_vals[-1])
    
    # detrend (or not)
    #I_detrended = I_reg - np.mean(I_reg)
    I_detrended = I_reg

    # PSD:
    fs = 1/dt  # sampling frequency (per year)
    freqs, psd = signal.welch(I_detrended, fs=fs, nperseg=len(I_reg))

    return freqs, psd

n_reps = 50
dt = 1/52  # weekly resolution (years)
transient = 30  # discard first n years

args = (beta0, gamma, mu, delta, dbeta,
        S0, I0, R0, T_max, N0, dt, transient)

# simulations in parallel
with Pool(processes=25) as pool: 
    results = pool.starmap(compute_psd_from_sim, [args for _ in range(n_reps)])

# average PSDs
freqs = results[0][0]  # common frequency grid
psd_matrix = np.vstack([psd for _, psd in results])
psd_mean = np.mean(psd_matrix, axis=0)

In [ ]:
for i in range(n_reps):
    plt.plot(results[i][0], results[i][1], color="black", alpha=0.2)
    plt.xlim([0.1, 0.5])

In [ ]:
# plot in period domain
periods = 1/(freqs + 1e-9)
mask = (freqs > 0.1) & (freqs < 1.0)
plt.figure(figsize=(8,5))
plt.plot(periods[mask], psd_mean[mask], color="navy", lw=2)
plt.scatter(periods[mask], psd_mean[mask])
plt.xlabel("Period (years)")
plt.ylabel("Mean PSD")
plt.title(f"Average PSD across {n_reps} Gillespie simulations")
plt.xlim([2, 8.333])
plt.grid(True)
plt.show()


In [ ]:
# to angular frequency
omegas = 2*np.pi*freqs
psd_omega = psd_mean / (2*np.pi)

# save to text file (for plotting with PSD notebook)
np.savetxt("psd_output.txt", np.column_stack([omegas, psd_omega]),
           header="omega(rad/year)   PSD(variance per rad/year)")
